In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import timm
import torch
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from torch import nn
from torch.optim import Adam
from torchvision import models
import splitfolders
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

import os
import shutil


In [ ]:
# cpu or gpu（cuda）
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)  # 确认输出的是 cuda:0
torch.cuda.init()


In [ ]:
# have already known
label_dic = {
    0: "Healthy",
    1: "Common Rust",
    2: "Blight",
    3: "Gray Leaf Spot",
}

In [ ]:
# read image names and labels of test and train sets
train_df = pd.read_csv('./data/train.csv')
test_df = pd.read_csv('./data/random_submission.csv')

In [ ]:
# divide images of train set into four floders following by "label_dic"
for label in label_dic.keys():
    if os.path.isdir("./train/{}".format(label_dic[label])):
        pass
    else:
        os.makedirs("./train/{}".format(label_dic[label]))
    #copy the images of source folder to these of new folder  
    for image_id in train_df[train_df["Label"]==label]["Image"]:
        shutil.copy('./data/train/{}.jpg'.format(image_id),'./train/{}/{}.jpg'.format(label_dic[label],image_id))

In [ ]:
# divide images of test set into four floders following by "label_dic"
for label in label_dic.keys():
    if os.path.isdir("./test/{}".format(label_dic[label])):
        pass
    else:
        os.makedirs('./test/{}'.format(label_dic[label]))
    for image_id in test_df[test_df["Predicted"]==label]["Image"]:
        shutil.copy('./data/test/{}.jpg'.format(image_id), './test/{}/{}.jpg'.format(label_dic[label], image_id))

In [ ]:
# need validation set from dividing train set(because test set dont have actual labels,just predicted labels)
splitfolders.ratio(
    "./train",
    output="splitted_data",
    seed=42,
    group_prefix=None,
    ratio=(.8, .2),
    move = False
)

define transform method of train data and val data and test data and all data. 

In [ ]:
# In addition to resizing to a fixed size, converting to PyTorch-compatible data, and normalization, additional transformation operations are required to achieve data augmentation.
train_transforms = transforms.Compose([
    transforms.Resize((256,256)),
# The image is randomly flipped horizontally with a 50% probability. This is a data augmentation technique that helps the model learn invariance to image orientation.
    transforms.RandomHorizontalFlip(),
# The image is randomly flipped vertically with a 50% probability. This is another data augmentation technique.
    transforms.RandomVerticalFlip(),
# Apply random Gaussian blur to the image. This added noise helps the model become more robust in predicting slight variations in the image.
    transforms.GaussianBlur(kernel_size=(3, 7), sigma=(0.1, 2)),# , sigma control indensity of Gaussian blur
# narray convert to PyTorch tensor
    transforms.ToTensor(),
# Normalize the tensor image using the mean and standard deviation.
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

上述train_transform包括数据增强（随机翻转，高斯模糊）但下面的Val_transform和test_transform是确定性的，没有随机转换，因为需要评估模型的性能，确保模型的预测结果稳定且可靠。如果使用随机增强，可能会导致测试结果不一致，影响模型性能的评估。
train_transform above has data augmentation, but Val_transform and test_transform below are ensured, not random transformations, ensure that the prediction results of the model are stable and reliable.If random enhancement is used, it may lead to inconsistent test results and affect the evaluation of model performance.

In [ ]:
val_transform = transforms.Compose([
    transforms.Resize(size=(256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize(size=(256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])  #官网给出的
])
# 这个对全部数据进行处理的图像管道并不对后续建模起作用，只对下面的可视化起作用
# This image pipeline, which processes all the data, does not work for subsequent modeling, only for the visualization below
data_transform = transforms.Compose([
    transforms.Resize(size=(256, 256)),
    transforms.ToTensor(),
])

## Loading the Data



In [ ]:
# 将一开始分类好的数据应用到上述定义好的图像转换管道
train = datasets.ImageFolder(root="./splitted_data/train",
                              transform=train_transforms)
# 自动识别子文件夹，并将每个子文件夹的名称视为类别标签。
# 遍历每个子文件夹，加载其中的图像，返回图像及其对应的类别标签。
# 支持应用自定义的预处理管道（通过 transform 参数）。
val = datasets.ImageFolder(root="./splitted_data/val",
                              transform=val_transform)

test = datasets.ImageFolder(root="./test",
                              transform=test_transform)

data = datasets.ImageFolder(root="./train",
                              transform=data_transform)
# print(train.samples)
# print(val.samples)
# print(test.samples)

In [ ]:
# 由于上述通过管道后的标签映射是按类别标签的首字母
# label_dic={
#     0: "Blight",
#     1: "Common Rust",
#     2: "Gray Leaf Spot",
#     3: "Healthy",
# }
# 来排序定义的，这跟我们数据中的标签映射
# label_dic={
#     0: "Healthy",
#     1: "Common Rust",
#     2: "Blight",
#     3: "Gray Leaf Spot",
# }
# 顺序并不相同，在这需要进行统一处理
print("Class to Index Mapping:", data.class_to_idx)
new_label_dic = {v: k for k, v in label_dic.items()}
for k, v in new_label_dic.items():
    print(k,v)
train.samples = [(path, new_label_dic[train.classes[label]]) for path, label in train.samples]
val.samples = [(path, new_label_dic[val.classes[label]]) for path, label in val.samples]
test.samples = [(path, new_label_dic[test.classes[label]]) for path, label in test.samples]

#### Plotting a few Images

Images plotted below did not undergo a data augmentation process. Therefore, they seem to be normal.

In [ ]:
# 对训练集中通过图像转换管道处理前的随机15张图片进行展示
labels_for_viz = {v: k for k, v in data.class_to_idx.items()}

fig, ax = plt.subplots(3, 5, figsize=(15, 10))
ax = ax.flatten()
for i in range(15):
    sample = random.randint(0, len(data))
    ax[i].imshow(data[sample][0].permute(1, 2, 0))
    ax[i].title.set_text(labels_for_viz[data[sample][1]])

### Plotting the Augmented Images

Because data augmentation methods are applied to Train set, it can be seen that images are blurred, horizontally and vertically flipped compared to the unaugmented images.

In [ ]:
# 对训练集中通过图像转换管道处理后的随机15张图片进行展示，与上述形成对比可方便看出图片经过管道处理变成了什么样子
fig, ax = plt.subplots(3, 5, figsize=(15, 10))
ax = ax.flatten()
for i in range(15):
    sample = random.randint(0, len(train))
    ax[i].imshow(train[sample][0].permute(1, 2, 0))
    ax[i].title.set_text(labels_for_viz[train[sample][1]])

在将数据输入神经网络之前，需要对数据进行 批量化（batching） 和 随机化（shuffling）。但是对于测试数据集，由于它不会参与模型的训练，因此没有必要对其进行随机化（shuffling）。

In [ ]:
# 对上述经过图像转化管道的数据按批次（每批次放64个数据）包装成数据加载器，训练和验证集可通过shuffle参数打乱顺序，测试集不参与模型训练验证不需要打乱顺序
#其实这里大部分的参数，起到的作用都是用空间去换时间(用多余的内存去换取更快的运行速度)(负载均衡)
train_dataloader = DataLoader(dataset=train,
                             batch_size=64,
                             num_workers=16,
                             shuffle=True)

val_dataloader = DataLoader(dataset=val,
                             batch_size=64,
                             num_workers=16,
                             shuffle=True)

test_dataloader = DataLoader(dataset=test,
                             batch_size=64,
                             num_workers=16,
                             shuffle=False)

## Building the Deep Learning Model

ResNet18

In [ ]:
# 加载在imagenet公开数据集上预先训练好的resnet模型
model = models.resnet18(pretrained=True)
# 更改预训练模型的输出层以匹配我们数据所需要预测的类的数量
model.fc = nn.Linear(in_features=512, out_features=4)
# 将模型移动到指定设备上计算
model = model.to(device)

## Defining the Loss Function and Choosing an Optimizer

In [ ]:
# 定义多分类的损失函数
loss_fn = nn.CrossEntropyLoss()
# 定义具有学习率的随机梯度下降优化器
optimizer = torch.optim.SGD(model.parameters(), lr=10e-3)

## Defining Train and Test Functions

In [ ]:
def train_and_validate_model(num_epochs):
    # 初始化模型性能的跟踪变量
    val_best_accuracy = 0.0
    train_loss_history = []
    val_loss_history = []
    train_accuracy_history = []
    val_accuracy_history = []
    
    print("Training begins...")
    for epoch in range(1, num_epochs+1):
        # 重置每个epoch的运行指标
        running_train_loss = 0.0
        running_train_accuracy = 0.0
        train_total = 0
        running_val_accuracy = 0.0
        running_val_loss = 0.0
        val_total = 0
        
        
        # TRAINING LOOP
        for data in tqdm(train_dataloader):
            inputs, outputs = data
            inputs, outputs = inputs.to(device), outputs.to(device)
            # 清除之前的梯度
            optimizer.zero_grad()
            # 前向传播
            predictions = model(inputs)
            # 计算损失
            train_loss = loss_fn(predictions, outputs)
            # 获取预测类别 忽略value 只要index， 这个会返回 value，index
            _, train_predicted = torch.max(predictions, 1)
            # 计算准确率
            running_train_accuracy += (train_predicted == outputs).sum().item()
            train_total += outputs.size(0) #0维数据，表示每个批次的样本数
            # 反向传播
            train_loss.backward()
            # 更新参数 optimizer.step() 让优化器根据当前计算的梯度（通过train_loss.backward() 计算），对模型的参数（例如权重和偏置）进行更新。
            optimizer.step()
            running_train_loss += train_loss.item()

        # 计算训练指标
        train_loss_value = running_train_loss/len(train_dataloader)
        train_loss_history.append(train_loss_value)
        train_accuracy = (100*running_train_accuracy)/train_total
        train_accuracy_history.append(train_accuracy)
        
        # VALIDATION LOOP
        # 禁用梯度计算
        with torch.no_grad():
            # 设置模型为评估模式
            model.eval()
            for data in val_dataloader:
                inputs, outputs = data
                inputs, outputs = inputs.to(device), outputs.to(device)
                predictions = model(inputs)
                val_loss = loss_fn(predictions, outputs)

                _, val_predicted = torch.max(predictions, 1)
                running_val_loss += val_loss.item()
                val_total += outputs.size(0)
                running_val_accuracy += (val_predicted == outputs).sum().item()

        # 计算验证指标
        val_loss_value = running_val_loss/len(test_dataloader)
        val_loss_history.append(val_loss_value)
        val_accuracy = (100*running_val_accuracy)/val_total
        val_accuracy_history.append(val_accuracy)

        # 保存最佳模型
        if val_accuracy > val_best_accuracy:
            torch.save(model.state_dict(), "model.pth")#把模型里面的参数保存为字典格式 模型参数（官方推荐）
            val_best_accuracy = val_accuracy

        # 打印每轮训练的详细信息
        print("Completed Epoch: ", epoch, "- Training Accuracy: %d" %train_accuracy, "- Validation Accuracy: %d" %val_accuracy, "- Training Loss: %.4f" %train_loss_value, "- Validation Loss: %.4f" %val_loss_value)
    #返回训练过程中的历史记录
    return train_accuracy_history, val_accuracy_history, train_loss_history, val_loss_history

In [ ]:
def test_model(): 
    # 重新构建模型结构，与训练时一致
    model = models.resnext101_64x4d(pretrained=True)
    model.fc = nn.Linear(in_features=2048, out_features=4,weight_decay=1e-5)
    model = model.to(device)
    path = "model.pth"
    # 对前面训练中保存的最优模型参数进行加载
    model.load_state_dict(torch.load(path))
    model.eval()

    # 初始化准确率和预测结果记录变量
    running_accuracy = 0
    total = 0
    all_predictions = []
    all_true = []
    
    with torch.no_grad():
        # 遍历验证数据集
        for data in val_dataloader:
            inputs, outputs = data
            # 将输入和标签移动到指定设备
            inputs, outputs = inputs.to(device), outputs.to(device)
            # 转换标签类型
            outputs = outputs.to(torch.float32)
            # 模型预测
            predictions = model(inputs)
            # 获取预测的类别
            _, predicted = torch.max(predictions, 1)

            # 计算准确率
            total += outputs.size(0)
            running_accuracy += (predicted == outputs).sum().item()
            accuracy = 100*running_accuracy/total

            # 收集预测结果和真实标签
            all_predictions.append(list(predicted.to("cpu").numpy()))
            all_true.append(list(outputs.to("cpu").numpy()))
         # 打印准确率
        print("Test Accuracy: ", accuracy)
    # 返回预测结果和真实标签
    return np.hstack(all_predictions), np.hstack(all_true)

In [ ]:
# 定义模型训练轮数
EPOCHS = 30

# 开始训练
train_accuracy_history, val_accuracy_history, train_loss_history, val_loss_history = train_and_validate_model(EPOCHS)
print("Training finished...\n")
all_preds = test_model()

In [ ]:
# 将验证集的数据应用到上述定义好的测试方法中进行预测
all_preds, all_true = test_model()

In [ ]:
# 绘制训练过程中的监控指标
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax = ax.flatten()

ax[0].plot(train_loss_history, label="Train")
ax[0].plot(val_loss_history, label="Validation")
ax[0].title.set_text("Loss")
ax[0].legend()

ax[1].plot(train_accuracy_history, label="Train")
ax[1].plot(val_accuracy_history, label="Validation")
ax[1].title.set_text("Accuracy Score")
ax[1].legend()

plt.show()

In [ ]:
# 查看模型在验证集上的预测分析报告
print(classification_report(all_true, all_preds, target_names=list(label_dic.values())))

In [ ]:
# 通过绘制混淆矩阵判断模型在验证集上的具体预测情况
plt.figure(figsize=(10, 7))
sns.heatmap(confusion_matrix(all_true, all_preds), cmap="Blues", annot=True, fmt="d", 
            xticklabels=list(label_dic.values()), yticklabels=list(label_dic.values()))
plt.title("Confusion Matrix", size=15)
plt.show()

In [ ]:
# 加载最优模型对测试集的数据进行预测并输出预测结果test_preds
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(in_features=512, out_features=4)
model = model.to(device)
path = "model.pth"
model.load_state_dict(torch.load(path))
model.eval()

all_predictions = []

with torch.no_grad():
    for data in test_dataloader:
        inputs, outputs = data
        inputs, outputs = inputs.to(device), outputs.to(device)
        outputs = outputs.to(torch.float32)
        predictions = model(inputs)
        _, predicted = torch.max(predictions, 1)
        
        all_predictions.append(list(predicted.to("cpu").numpy()))
test_preds = np.hstack(all_predictions)

In [ ]:
# 将上述的测试集预测结果test_preds按照random_submission.csv的格式进行整理
submission = pd.DataFrame([[int(_[0].split('/')[-1].split('.')[0]) for _ in test.samples], test_preds]).T.sort_values(0)

In [ ]:
# 整理输出提交结果文件submission.csv
submission.columns = ["Image", "Predicted"]
submission.to_csv('submission.csv', index=False)